# 01 - Veri setlerini indir ve doğrula

Bu not defteri iki veri setini Colab'ın geçici diskine indirir ve sayıları kontrol eder.

- **LUAD-HistoSeg**: akciğer, bitirme projesindeki veri seti
- **BCSS-WSSS**: meme, yeni ana veri seti

Kaynak: veri setlerini yayınlayan ekibin GitHub deposundaki resmî Google Drive bağlantıları
(https://github.com/ChuHan89/WSSS-Tissue).

Bu not defteri **model eğitmez**, GPU istemez. Sadece indirir, açar, sayar.

## 1. İndirme aracı

`gdown`, Google Drive'daki herkese açık klasörleri komutla indiren küçük bir araçtır.

In [ ]:
!pip -q install -U gdown
!gdown --version

## 2. LUAD-HistoSeg'i indir

- `--folder`: bağlantı tek dosya değil, klasör.
- `-O`: nereye kaydedileceği.

In [ ]:
!mkdir -p /content/data/indirilen/LUAD-HistoSeg
!gdown --folder "https://drive.google.com/drive/folders/1E3Yei3Or3xJXukHIybZAgochxfn6FJpr" -O /content/data/indirilen/LUAD-HistoSeg

## 3. BCSS-WSSS'i indir

In [ ]:
!mkdir -p /content/data/indirilen/BCSS-WSSS
!gdown --folder "https://drive.google.com/drive/folders/1iS2Z0DsbACqGp7m6VDJbAcgzeXNEFr77" -O /content/data/indirilen/BCSS-WSSS

## 4. Ne indi?

İnen dosyaların adını ve boyutunu listeler. Arşiv (zip/rar/7z) varsa bir sonraki hücre açar.

In [ ]:
!find /content/data/indirilen -maxdepth 3 -type f -printf "%s bayt  %p\n" | sort -k3 | head -100
!du -sh /content/data/indirilen/*

## 5. Arşivleri aç

Her arşivi, kendi veri setinin adını taşıyan klasöre açar: `/content/data/acik/<veri seti>/`.

In [ ]:
import subprocess
from pathlib import Path

INDIRILEN = Path("/content/data/indirilen")
ACIK = Path("/content/data/acik")

# rar ve 7z dosyalarını açabilmek için 7zip kurulur (zip için de çalışır).
subprocess.run("apt-get -qq install -y p7zip-full unrar > /dev/null 2>&1", shell=True)

for veri_seti in sorted(p for p in INDIRILEN.iterdir() if p.is_dir()):
    hedef = ACIK / veri_seti.name
    hedef.mkdir(parents=True, exist_ok=True)
    arsivler = [p for p in veri_seti.rglob("*") if p.suffix.lower() in {".zip", ".rar", ".7z"}]
    if not arsivler:
        print(veri_seti.name, ": arşiv yok, dosyalar zaten açık")
        continue
    for arsiv in arsivler:
        print("açılıyor:", arsiv.name)
        if arsiv.suffix.lower() == ".rar":
            komut = ["unrar", "x", "-o+", "-inul", str(arsiv), str(hedef) + "/"]
        else:
            komut = ["7z", "x", "-y", "-bso0", "-bsp0", f"-o{hedef}", str(arsiv)]
        sonuc = subprocess.run(komut, capture_output=True, text=True)
        print("  bitti, çıkış kodu:", sonuc.returncode, sonuc.stderr[:300])

## 6. Say ve doğrula

Her klasörde kaç görüntü olduğunu sayar. Beklenen sayılar (makalelerdeki değerler):

| Veri seti | eğitim | doğrulama | test |
|---|---|---|---|
| LUAD-HistoSeg | 16.678 | 300 (bazı makalelerde 306) | 307 |
| BCSS-WSSS | 23.422 | 3.418 | 4.986 |

In [ ]:
from collections import Counter
from pathlib import Path

RESIM = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
KOK = Path("/content/data/acik")


def gercek_resim(dosya):
    # "._ad.png" dosyaları ve "__MACOSX" klasörü, zip Mac'te hazırlandığı için gelen çöp kayıtlardır; resim değildir.
    return (
        dosya.is_file()
        and dosya.suffix.lower() in RESIM
        and not dosya.name.startswith("._")
        and "__MACOSX" not in dosya.parts
    )


sayim = Counter()
for dosya in KOK.rglob("*"):
    if gercek_resim(dosya):
        sayim[str(dosya.parent)] += 1

for klasor in sorted(sayim):
    print(f"{sayim[klasor]:>7}  {klasor}")

## 7. Etiketlere ve görüntülere bak

- Eğitim görüntülerinde etiket dosya adının içindedir: `ad-[1 0 0 1].png`. Köşeli parantezdeki 4 sayı, 4 doku sınıfından hangilerinin o yamada bulunduğunu söyler.
- Doğrulama ve test tarafında her görüntünün piksel piksel boyanmış bir maskesi vardır.

In [ ]:
import re
import numpy as np
from collections import Counter
from pathlib import Path
from PIL import Image

# Etiket iki biçimde yazılmış: LUAD'da "[1 0 0 1]", BCSS'te "[1001]". Desen ikisini de yakalar.
desen = re.compile(r"\[([01 ]+)\]")

for klasor in sorted(sayim):
    yol = Path(klasor)
    dosyalar = sorted(p for p in yol.iterdir() if gercek_resim(p))
    ornek = dosyalar[0]
    with Image.open(ornek) as im:
        boyut, mod = im.size, im.mode
    print("\n==", klasor)
    print("   örnek dosya:", ornek.name, "| boyut:", boyut, "| renk modu:", mod)

    eslesen = [desen.search(p.name) for p in dosyalar]
    etiketler = [[int(h) for h in m.group(1) if h in "01"] for m in eslesen if m]
    if etiketler:
        dizi = np.array(etiketler)
        print("   dosya adında etiket olan:", len(etiketler))
        print("   sınıf başına yama sayısı:", dizi.sum(axis=0).tolist())
        print("   yamadaki sınıf sayısı dağılımı:", dict(sorted(Counter(dizi.sum(axis=1).tolist()).items())))
    elif yol.name == "mask":
        degerler = Counter()
        for p in dosyalar:
            with Image.open(p) as im:
                degerler.update(np.unique(np.array(im)).tolist())
        print("   piksel değeri -> kaç maskede görüldü:", dict(sorted(degerler.items())))

print("\n== Görüntü ve maske adları birebir eşleşiyor mu?")
for klasor in sorted(sayim):
    yol = Path(klasor)
    if yol.name == "img":
        goruntuler = {p.stem for p in yol.iterdir() if gercek_resim(p)}
        maskeler = {p.stem for p in (yol.parent / "mask").iterdir() if gercek_resim(p)}
        print("  ", yol.parent, "| maskesiz görüntü:", len(goruntuler - maskeler), "| görüntüsüz maske:", len(maskeler - goruntuler))